# Phase 1: Exploratory Data Analysis

## Prediction and Maintenance of Critical Automotive Component Failure Using Machine Learning

This notebook performs the Phase 1 exploratory data analysis (EDA) for the vehicle telemetry dataset.

**Scope control:** this notebook does not train Logistic Regression, Random Forest, XGBoost, or any other predictive model. It does not remove observations or perform feature engineering. The purpose is to understand the data and document modelling implications.

## 1. Setup and Data Loading

The notebook is written to run from either the project root or the `notebooks/` directory. Figures are saved to `outputs/figures/eda/`.

In [ ]:
from pathlib import Path
from html import escape
import math

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Resources" / "synthetic_telemetry_data.xlsx").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "Resources" / "synthetic_telemetry_data.xlsx"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures" / "eda"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_PATH, sheet_name="Sheet1")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Figure directory: {FIGURE_DIR}")
print(f"Loaded dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

## Plotting Helpers

These lightweight helpers save SVG figures without requiring model or plotting-library dependencies.

In [ ]:
def _svg_text(x, y, text, size=12, anchor="start", weight="normal", rotate=None, fill="#222"):
    text = escape(str(text))
    transform = f' transform="rotate({rotate} {x} {y})"' if rotate is not None else ""
    return (
        f'<text x="{x:.2f}" y="{y:.2f}" font-size="{size}" '
        f'font-family="Arial, Helvetica, sans-serif" text-anchor="{anchor}" '
        f'font-weight="{weight}" fill="{fill}"{transform}>{text}</text>'
    )


def _save_svg(path, width, height, body):
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">'
        '<rect width="100%" height="100%" fill="white"/>'
        f"{body}</svg>"
    )
    Path(path).write_text(svg, encoding="utf-8")


def _fmt(value, digits=2):
    if pd.isna(value):
        return "NA"
    if isinstance(value, (int, np.integer)):
        return f"{value:,}"
    return f"{float(value):,.{digits}f}"


def save_horizontal_bar(labels, values, path, title, xlabel="Count", color="#4472C4"):
    labels = [str(x) for x in labels]
    values = [float(x) for x in values]
    width = 1000
    row_h = 30
    margin_l = 250
    margin_r = 80
    margin_t = 70
    margin_b = 55
    height = margin_t + margin_b + row_h * max(1, len(labels))
    plot_w = width - margin_l - margin_r
    max_v = max(values) if values else 1
    max_v = max(max_v, 1)
    parts = [_svg_text(width / 2, 32, title, 18, anchor="middle", weight="bold")]
    parts.append(f'<line x1="{margin_l}" y1="{height-margin_b}" x2="{width-margin_r}" y2="{height-margin_b}" stroke="#333"/>')
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_t + i * row_h
        bar_w = plot_w * value / max_v
        parts.append(_svg_text(margin_l - 10, y + 18, label, 12, anchor="end"))
        parts.append(f'<rect x="{margin_l}" y="{y}" width="{bar_w:.2f}" height="20" fill="{color}" opacity="0.85"/>')
        parts.append(_svg_text(margin_l + bar_w + 6, y + 15, _fmt(value, 0), 11))
    parts.append(_svg_text(margin_l + plot_w / 2, height - 18, xlabel, 12, anchor="middle"))
    _save_svg(path, width, height, "".join(parts))


def save_line_chart(labels, values, path, title, ylabel="Count", color="#4472C4"):
    labels = [str(x) for x in labels]
    values = [float(x) for x in values]
    width = 1100
    height = 420
    ml, mr, mt, mb = 80, 40, 65, 80
    plot_w = width - ml - mr
    plot_h = height - mt - mb
    y_max = max(values) if values else 1
    y_max = max(y_max, 1)
    points = []
    for i, value in enumerate(values):
        x = ml + (plot_w * i / max(1, len(values) - 1))
        y = mt + plot_h - (plot_h * value / y_max)
        points.append((x, y))
    parts = [_svg_text(width / 2, 32, title, 18, anchor="middle", weight="bold")]
    parts.append(f'<line x1="{ml}" y1="{mt}" x2="{ml}" y2="{mt+plot_h}" stroke="#333"/>')
    parts.append(f'<line x1="{ml}" y1="{mt+plot_h}" x2="{ml+plot_w}" y2="{mt+plot_h}" stroke="#333"/>')
    for frac in [0, 0.25, 0.5, 0.75, 1]:
        y = mt + plot_h - frac * plot_h
        val = frac * y_max
        parts.append(f'<line x1="{ml}" y1="{y:.2f}" x2="{ml+plot_w}" y2="{y:.2f}" stroke="#e6e6e6"/>')
        parts.append(_svg_text(ml - 8, y + 4, _fmt(val, 0), 10, anchor="end"))
    if len(points) > 1:
        point_str = " ".join(f"{x:.2f},{y:.2f}" for x, y in points)
        parts.append(f'<polyline points="{point_str}" fill="none" stroke="{color}" stroke-width="3"/>')
    for x, y in points:
        parts.append(f'<circle cx="{x:.2f}" cy="{y:.2f}" r="3" fill="{color}"/>')
    if labels:
        label_positions = sorted(set([0, len(labels)//2, len(labels)-1]))
        for idx in label_positions:
            x = ml + (plot_w * idx / max(1, len(labels) - 1))
            parts.append(_svg_text(x, height - 45, labels[idx], 10, anchor="middle", rotate=0))
    parts.append(_svg_text(18, mt + plot_h / 2, ylabel, 12, anchor="middle", rotate=-90))
    _save_svg(path, width, height, "".join(parts))


def save_histogram_grid(data, features, path, title, bins=18):
    features = [f for f in features if f in data.columns]
    cols = 2
    rows = math.ceil(len(features) / cols)
    panel_w = 520
    panel_h = 260
    width = cols * panel_w
    height = 60 + rows * panel_h
    parts = [_svg_text(width / 2, 30, title, 18, anchor="middle", weight="bold")]
    for idx, feature in enumerate(features):
        r, c = divmod(idx, cols)
        x0 = c * panel_w + 70
        y0 = 65 + r * panel_h
        plot_w = panel_w - 110
        plot_h = panel_h - 85
        series = pd.to_numeric(data[feature], errors="coerce").dropna()
        if series.empty:
            continue
        counts, edges = np.histogram(series, bins=bins)
        max_count = max(counts.max(), 1)
        parts.append(_svg_text(x0 + plot_w / 2, y0 + 8, feature, 13, anchor="middle", weight="bold"))
        parts.append(f'<line x1="{x0}" y1="{y0+plot_h}" x2="{x0+plot_w}" y2="{y0+plot_h}" stroke="#333"/>')
        parts.append(f'<line x1="{x0}" y1="{y0}" x2="{x0}" y2="{y0+plot_h}" stroke="#333"/>')
        bar_w = plot_w / len(counts)
        for i, count in enumerate(counts):
            h = plot_h * count / max_count
            x = x0 + i * bar_w
            y = y0 + plot_h - h
            parts.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{max(1, bar_w-1):.2f}" height="{h:.2f}" fill="#70AD47" opacity="0.82"/>')
        parts.append(_svg_text(x0, y0 + plot_h + 20, _fmt(series.min(), 2), 10, anchor="start"))
        parts.append(_svg_text(x0 + plot_w, y0 + plot_h + 20, _fmt(series.max(), 2), 10, anchor="end"))
    _save_svg(path, width, height, "".join(parts))


def save_boxplot_grid(data, features, path, title):
    features = [f for f in features if f in data.columns]
    width = 1100
    row_h = 58
    ml, mr, mt, mb = 260, 60, 70, 40
    height = mt + mb + row_h * max(1, len(features))
    plot_w = width - ml - mr
    parts = [_svg_text(width / 2, 32, title, 18, anchor="middle", weight="bold")]
    for idx, feature in enumerate(features):
        y = mt + idx * row_h + 25
        series = pd.to_numeric(data[feature], errors="coerce").dropna()
        if series.empty:
            continue
        mn, q1, med, q3, mx = np.percentile(series, [0, 25, 50, 75, 100])
        def sx(v):
            if mx == mn:
                return ml + plot_w / 2
            return ml + (float(v) - mn) / (mx - mn) * plot_w
        parts.append(_svg_text(ml - 12, y + 4, feature, 12, anchor="end"))
        parts.append(f'<line x1="{sx(mn):.2f}" y1="{y}" x2="{sx(mx):.2f}" y2="{y}" stroke="#666" stroke-width="2"/>')
        parts.append(f'<rect x="{sx(q1):.2f}" y="{y-12}" width="{max(1, sx(q3)-sx(q1)):.2f}" height="24" fill="#ED7D31" opacity="0.75" stroke="#333"/>')
        parts.append(f'<line x1="{sx(med):.2f}" y1="{y-15}" x2="{sx(med):.2f}" y2="{y+15}" stroke="#111" stroke-width="2"/>')
        parts.append(_svg_text(sx(mn), y + 30, _fmt(mn, 1), 9, anchor="middle"))
        parts.append(_svg_text(sx(mx), y + 30, _fmt(mx, 1), 9, anchor="middle"))
    _save_svg(path, width, height, "".join(parts))


def save_standardized_mean_difference(data, target, features, path, title):
    rows = []
    for feature in features:
        if feature not in data.columns:
            continue
        s = pd.to_numeric(data[feature], errors="coerce")
        std = s.std()
        if std == 0 or pd.isna(std):
            diff = 0
        else:
            diff = (data.loc[data[target] == 1, feature].mean() - data.loc[data[target] == 0, feature].mean()) / std
        rows.append((feature, float(diff)))
    rows = sorted(rows, key=lambda x: abs(x[1]), reverse=True)
    width = 1050
    row_h = 34
    ml, mr, mt, mb = 300, 70, 70, 60
    height = mt + mb + row_h * max(1, len(rows))
    plot_w = width - ml - mr
    max_abs = max([abs(v) for _, v in rows] + [1])
    center = ml + plot_w / 2
    parts = [_svg_text(width / 2, 32, title, 18, anchor="middle", weight="bold")]
    parts.append(f'<line x1="{center:.2f}" y1="{mt-10}" x2="{center:.2f}" y2="{height-mb+10}" stroke="#333"/>')
    for i, (feature, diff) in enumerate(rows):
        y = mt + i * row_h
        bar_w = (plot_w / 2) * abs(diff) / max_abs
        x = center if diff >= 0 else center - bar_w
        color = "#C00000" if diff >= 0 else "#4472C4"
        parts.append(_svg_text(ml - 12, y + 18, feature, 12, anchor="end"))
        parts.append(f'<rect x="{x:.2f}" y="{y}" width="{bar_w:.2f}" height="22" fill="{color}" opacity="0.82"/>')
        parts.append(_svg_text(x + (bar_w + 6 if diff >= 0 else -6), y + 16, _fmt(diff, 2), 10, anchor=("start" if diff >= 0 else "end")))
    parts.append(_svg_text(center, height - 22, "Standardized mean difference: target=1 minus target=0", 12, anchor="middle"))
    _save_svg(path, width, height, "".join(parts))


def save_correlation_heatmap(corr, path, title):
    labels = list(corr.columns)
    n = len(labels)
    cell = 34
    ml, mt = 240, 95
    width = ml + n * cell + 60
    height = mt + n * cell + 260
    parts = [_svg_text(width / 2, 32, title, 18, anchor="middle", weight="bold")]
    def color_for(v):
        v = max(-1, min(1, float(v)))
        if v >= 0:
            intensity = int(255 - 135 * v)
            return f"rgb(255,{intensity},{intensity})"
        intensity = int(255 - 135 * abs(v))
        return f"rgb({intensity},{intensity},255)"
    for i, label in enumerate(labels):
        parts.append(_svg_text(ml - 8, mt + i * cell + 22, label, 10, anchor="end"))
        parts.append(_svg_text(ml + i * cell + 16, mt - 12, label, 10, anchor="end", rotate=-45))
    for r, row_label in enumerate(labels):
        for c, col_label in enumerate(labels):
            v = corr.loc[row_label, col_label]
            parts.append(f'<rect x="{ml+c*cell}" y="{mt+r*cell}" width="{cell}" height="{cell}" fill="{color_for(v)}" stroke="white"/>')
            if abs(v) >= 0.5 or r == c:
                parts.append(_svg_text(ml+c*cell+cell/2, mt+r*cell+cell/2+4, f"{v:.1f}", 8, anchor="middle"))
    parts.append(_svg_text(width / 2, height - 45, "Red = positive correlation, blue = negative correlation", 12, anchor="middle"))
    _save_svg(path, width, height, "".join(parts))


print("SVG plotting helpers ready.")

## 2. Dataset Overview

This section documents shape, columns, data types, sample records, descriptive statistics, and approximate memory usage.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.to_list())

print("\nData types:")
print(df.dtypes)

print("\nSample observations:")
print(df.head(5).to_string(index=False))

print("\nDescriptive statistics for numerical variables:")
print(df.describe(include=[np.number]).T.round(3).to_string())

print("\nCategorical summary:")
print(df.describe(include=["object"]).T.to_string())

memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nApproximate memory usage: {memory_mb:.3f} MB")

## 3. Data Quality

This section checks for missing values, duplicate records, constant and near-constant columns, suspicious values, and categorical consistency. Findings are documented only; no rows or columns are removed.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
print(missing.to_string())

full_duplicate_rows = int(df.duplicated().sum())
duplicate_vehicle_timestamp = int(df.duplicated(["vehicle_id", "timestamp"]).sum())
print(f"\nFull duplicate rows: {full_duplicate_rows}")
print(f"Duplicate vehicle_id + timestamp pairs: {duplicate_vehicle_timestamp}")

constant_columns = []
near_constant_columns = []
for column in df.columns:
    nunique = df[column].nunique(dropna=False)
    top_share = df[column].value_counts(dropna=False, normalize=True).iloc[0]
    if nunique == 1:
        constant_columns.append(column)
    elif top_share >= 0.95:
        near_constant_columns.append((column, nunique, round(float(top_share), 4)))

print("\nConstant columns:", constant_columns)
print("Near-constant columns at >=95% same value:")
print(near_constant_columns)

percent_cols = [
    "fuel_level_percent", "engine_load_percent", "throttle_pos_percent",
    "brake_pedal_pos_percent", "battery_charge_percent",
    "battery_health_percent", "humidity_percent"
]
nonnegative_cols = [
    "engine_rpm", "oil_pressure_psi", "brake_fluid_level_psi",
    "brake_pad_wear_mm", "vehicle_speed_kph", "battery_voltage_v",
    "alternator_output_v"
]
wheel_speed_cols = [
    "wheel_speed_fl_kph", "wheel_speed_fr_kph",
    "wheel_speed_rl_kph", "wheel_speed_rr_kph"
]

suspicious = {}
suspicious["percent_outside_0_100"] = {
    c: int(((df[c] < 0) | (df[c] > 100)).sum()) for c in percent_cols
}
suspicious["nonnegative_below_zero"] = {
    c: int((df[c] < 0).sum()) for c in nonnegative_cols
}
suspicious["wheel_speed_below_zero"] = {
    c: int((df[c] < 0).sum()) for c in wheel_speed_cols
}
suspicious["gps_outside_valid_range"] = {
    "gps_latitude": int(((df["gps_latitude"] < -90) | (df["gps_latitude"] > 90)).sum()),
    "gps_longitude": int(((df["gps_longitude"] < -180) | (df["gps_longitude"] > 180)).sum()),
}
suspicious["vibration_below_zero"] = int((df["vibration_level"] < 0).sum())

print("\nSuspicious value checks:")
for group, values in suspicious.items():
    print(group, values)

print("\nCategorical values:")
for column in df.select_dtypes(include=["object"]).columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False).to_string())

**Data-quality notes:** the initial inspection found no missing values and no full duplicate rows. There are duplicate `vehicle_id` + `timestamp` pairs, and several target/outcome fields are near-constant because failure events are rare. A small number of wheel-speed values are negative, humidity values exceed 100 in a few records, and one vibration value is slightly negative; these should be investigated during EDA before any cleaning decisions are made.

## 4. Vehicle Structure

This section checks vehicle IDs, observations per vehicle, brand frequencies, repeated telemetry observations, and timestamp ordering.

In [ ]:
vehicle_counts = df["vehicle_id"].value_counts().sort_index()
brand_counts = df["brand"].value_counts()
obs_per_vehicle_summary = vehicle_counts.describe()

print(f"Unique vehicle IDs: {df['vehicle_id'].nunique()}")
print("\nObservations per vehicle summary:")
print(obs_per_vehicle_summary.round(3).to_string())
print("\nVehicle brand frequencies:")
print(brand_counts.to_string())

repeated_vehicles = int((vehicle_counts > 1).sum())
print(f"\nVehicles with repeated telemetry observations: {repeated_vehicles} of {df['vehicle_id'].nunique()}")

non_monotonic_counts = {}
for vehicle_id, group in df.groupby("vehicle_id", sort=False):
    non_monotonic_counts[vehicle_id] = int((group["timestamp"].diff().dt.total_seconds() < 0).sum())
vehicles_with_non_monotonic_file_order = sum(v > 0 for v in non_monotonic_counts.values())
print(f"Vehicles with non-monotonic timestamp order in file order: {vehicles_with_non_monotonic_file_order}")
print(f"Total non-monotonic transitions in file order: {sum(non_monotonic_counts.values())}")

save_horizontal_bar(
    brand_counts.index,
    brand_counts.values,
    FIGURE_DIR / "brand_frequency.svg",
    "Vehicle Brand Frequency",
    xlabel="Records"
)
save_horizontal_bar(
    vehicle_counts.index,
    vehicle_counts.values,
    FIGURE_DIR / "observations_per_vehicle.svg",
    "Observations per Vehicle",
    xlabel="Records",
    color="#70AD47"
)
print("\nSaved vehicle structure figures:")
print(FIGURE_DIR / "brand_frequency.svg")
print(FIGURE_DIR / "observations_per_vehicle.svg")

## 5. Temporal Analysis

Repeated observations per vehicle imply temporal dependence. Random row-level splitting can leak information because measurements from the same vehicle, and potentially adjacent time periods, may appear in both train and test sets.

In [ ]:
print(f"Timestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")

sorted_by_vehicle_time = df.sort_values(["vehicle_id", "timestamp"])
interval_minutes = (
    sorted_by_vehicle_time
    .groupby("vehicle_id")["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)
print("\nReading interval summary after sorting within vehicle (minutes):")
print(interval_minutes.dropna().describe().round(3).to_string())

hourly_counts = df.set_index("timestamp").sort_index().resample("h").size()
print("\nObservations by hour:")
print(hourly_counts.to_string())

save_line_chart(
    [idx.strftime("%Y-%m-%d %H:%M") for idx in hourly_counts.index],
    hourly_counts.values,
    FIGURE_DIR / "observations_over_time.svg",
    "Telemetry Observations Over Time",
    ylabel="Records"
)
print("\nSaved temporal figure:")
print(FIGURE_DIR / "observations_over_time.svg")

**Temporal split implication:** because each vehicle has repeated records, a random row-level split could place the same vehicle in both training and test data. This can inflate performance estimates through vehicle-specific memorization and short-range temporal similarity. Grouped validation by `vehicle_id`, and possibly time-aware validation, should be considered before modelling.

## 6. Target Analysis

The failure-related fields are analyzed carefully before defining the modelling target. No final target formulation is selected in this notebook.

In [ ]:
target_cols = [
    "engine_failure_imminent",
    "brake_issue_imminent",
    "battery_issue_imminent",
]

target_rows = []
for target in target_cols:
    counts = df[target].value_counts().sort_index()
    neg = int(counts.get(0, 0))
    pos = int(counts.get(1, 0))
    target_rows.append({
        "target": target,
        "negative_count": neg,
        "positive_count": pos,
        "negative_percent": neg / len(df) * 100,
        "positive_percent": pos / len(df) * 100,
        "negative_to_positive_ratio": np.nan if pos == 0 else neg / pos,
    })
target_summary = pd.DataFrame(target_rows)
print("Imminent-failure target summary:")
print(target_summary.round(3).to_string(index=False))

any_issue = df[target_cols].max(axis=1)
print("\nAny imminent issue counts:")
print(any_issue.value_counts().sort_index().to_string())
print(f"Rows with multiple imminent issue flags: {int((df[target_cols].sum(axis=1) > 1).sum())}")

save_horizontal_bar(
    target_summary["target"],
    target_summary["positive_count"],
    FIGURE_DIR / "target_positive_counts.svg",
    "Positive Class Count by Imminent-Failure Target",
    xlabel="Positive records",
    color="#C00000"
)

failure_type_counts = df["failure_type"].value_counts()
print("\nFailure type frequency:")
print(failure_type_counts.to_string())
save_horizontal_bar(
    failure_type_counts.index,
    failure_type_counts.values,
    FIGURE_DIR / "failure_type_frequency.svg",
    "Failure Type Frequency",
    xlabel="Records",
    color="#7030A0"
)

print("\nRelationship between failure_type and imminent-failure indicators:")
failure_flag_crosstab = pd.crosstab(
    df["failure_type"],
    [df["engine_failure_imminent"], df["brake_issue_imminent"], df["battery_issue_imminent"]],
    rownames=["failure_type"],
    colnames=["engine", "brake", "battery"],
)
print(failure_flag_crosstab.to_string())

no_failure_sentinel = pd.Timestamp("2050-01-01")
print("\nFailure date summary:")
print(df["failure_date"].describe().to_string())
print(f"Records with failure_date = 2050-01-01: {int((df['failure_date'] == no_failure_sentinel).sum())}")
actual_failure_dates = df.loc[df["failure_date"] != no_failure_sentinel, "failure_date"]
print(f"Non-sentinel failure_date range: {actual_failure_dates.min()} to {actual_failure_dates.max()}")

print("\nSaved target analysis figures:")
print(FIGURE_DIR / "target_positive_counts.svg")
print(FIGURE_DIR / "failure_type_frequency.svg")

**Failure-date interpretation:** `failure_date` appears to contain future outcome timing or a no-failure sentinel value (`2050-01-01`). It should not be used directly as a predictor for imminent-failure classification unless a later time-to-event formulation is explicitly designed and leakage-safe.

## 7. Feature Groups

The variables are organized into logical groups for EDA. These groups support interpretation, but they do not imply that each group must only be used for one target.

In [ ]:
feature_groups = {
    "identifiers_time": ["vehicle_id", "brand", "timestamp", "failure_date"],
    "engine": [
        "engine_temp_c", "engine_rpm", "oil_pressure_psi", "coolant_temp_c",
        "fuel_level_percent", "fuel_consumption_lph", "engine_load_percent",
        "throttle_pos_percent", "air_flow_rate_gps", "exhaust_gas_temp_c",
        "vibration_level", "engine_hours"
    ],
    "braking_system": [
        "brake_fluid_level_psi", "brake_pad_wear_mm", "brake_temp_c",
        "abs_fault_indicator", "brake_pedal_pos_percent",
        "wheel_speed_fl_kph", "wheel_speed_fr_kph",
        "wheel_speed_rl_kph", "wheel_speed_rr_kph"
    ],
    "battery_electrical": [
        "battery_voltage_v", "battery_current_a", "battery_temp_c",
        "alternator_output_v", "battery_charge_percent", "battery_health_percent"
    ],
    "vehicle_operation": ["odometer_reading", "vehicle_speed_kph"],
    "environmental": ["ambient_temp_c", "humidity_percent"],
    "location_gps": ["gps_latitude", "gps_longitude"],
    "target_outcome_fields": [
        "engine_failure_imminent", "brake_issue_imminent",
        "battery_issue_imminent", "failure_type"
    ],
}

for group_name, columns in feature_groups.items():
    print(f"\n{group_name}:")
    print(columns)

## 8. Univariate Analysis

The following plots focus on a selected set of informative telemetry variables rather than generating redundant plots for every column.

In [ ]:
selected_univariate_features = [
    "engine_temp_c", "engine_rpm", "oil_pressure_psi", "vibration_level",
    "brake_fluid_level_psi", "brake_temp_c",
    "battery_voltage_v", "battery_current_a", "battery_health_percent",
    "vehicle_speed_kph"
]

range_summary = df[selected_univariate_features].agg(["min", "median", "mean", "max", "std"]).T
print("Selected telemetry range summary:")
print(range_summary.round(3).to_string())

outlier_rows = []
for feature in selected_univariate_features:
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_rows.append({
        "feature": feature,
        "lower_iqr_bound": lower,
        "upper_iqr_bound": upper,
        "iqr_outlier_count": int(((df[feature] < lower) | (df[feature] > upper)).sum()),
    })
outlier_summary = pd.DataFrame(outlier_rows)
print("\nIQR-based potential outlier counts for selected telemetry variables:")
print(outlier_summary.round(3).to_string(index=False))

save_histogram_grid(
    df,
    selected_univariate_features,
    FIGURE_DIR / "selected_telemetry_histograms.svg",
    "Selected Telemetry Distributions"
)
save_boxplot_grid(
    df,
    selected_univariate_features,
    FIGURE_DIR / "selected_telemetry_boxplots.svg",
    "Selected Telemetry Ranges and Boxplots"
)
print("\nSaved univariate figures:")
print(FIGURE_DIR / "selected_telemetry_histograms.svg")
print(FIGURE_DIR / "selected_telemetry_boxplots.svg")

## 9. Failure vs Non-Failure Analysis

For each imminent-failure target, selected telemetry measurements are compared between target value 0 and target value 1.

In [ ]:
component_target_features = {
    "engine_failure_imminent": [
        "engine_temp_c", "engine_rpm", "oil_pressure_psi", "coolant_temp_c",
        "engine_load_percent", "exhaust_gas_temp_c", "vibration_level"
    ],
    "brake_issue_imminent": [
        "brake_fluid_level_psi", "brake_pad_wear_mm", "brake_temp_c",
        "abs_fault_indicator", "brake_pedal_pos_percent",
        "wheel_speed_fl_kph", "wheel_speed_fr_kph",
        "wheel_speed_rl_kph", "wheel_speed_rr_kph"
    ],
    "battery_issue_imminent": [
        "battery_voltage_v", "battery_current_a", "battery_temp_c",
        "alternator_output_v", "battery_charge_percent", "battery_health_percent"
    ],
}

for target, features in component_target_features.items():
    print(f"\n=== {target} ===")
    summary = df.groupby(target)[features].agg(["count", "mean", "median", "std", "min", "max"])
    print(summary.round(3).to_string())

    filename = target.replace("_imminent", "") + "_standardized_mean_differences.svg"
    save_standardized_mean_difference(
        df,
        target,
        features,
        FIGURE_DIR / filename,
        f"{target}: Standardized Mean Differences"
    )
    print("Saved:", FIGURE_DIR / filename)

**Failure-pattern notes from EDA:** engine-positive rows show higher engine temperature and vibration and much lower oil pressure than engine-negative rows. Brake-positive rows show lower brake fluid level and higher brake temperature than brake-negative rows. Battery-positive rows show lower battery voltage and strongly negative battery current compared with battery-negative rows. These are descriptive EDA patterns only, not model findings.

## 10. Correlation Analysis

A selected set of numerical telemetry variables is used for the heatmap to keep it readable. Target and outcome fields are excluded.

In [ ]:
corr_features = [
    "odometer_reading", "engine_hours", "engine_temp_c", "engine_rpm",
    "oil_pressure_psi", "coolant_temp_c", "engine_load_percent",
    "exhaust_gas_temp_c", "vibration_level", "brake_fluid_level_psi",
    "brake_pad_wear_mm", "brake_temp_c", "abs_fault_indicator",
    "battery_voltage_v", "battery_current_a", "battery_temp_c",
    "alternator_output_v", "battery_charge_percent", "battery_health_percent",
    "vehicle_speed_kph", "ambient_temp_c", "humidity_percent"
]
corr = df[corr_features].corr(numeric_only=True)
save_correlation_heatmap(
    corr,
    FIGURE_DIR / "selected_numeric_correlation_heatmap.svg",
    "Selected Numerical Telemetry Correlation Heatmap"
)

corr_abs = corr.abs()
pairs = []
for i, a in enumerate(corr_abs.columns):
    for b in corr_abs.columns[i + 1:]:
        pairs.append((a, b, corr.loc[a, b], corr_abs.loc[a, b]))
top_pairs = (
    pd.DataFrame(pairs, columns=["feature_a", "feature_b", "correlation", "abs_correlation"])
    .sort_values("abs_correlation", ascending=False)
    .head(15)
)
print("Top absolute correlations among selected numerical telemetry variables:")
print(top_pairs.round(3).to_string(index=False))
print("\nSaved correlation heatmap:")
print(FIGURE_DIR / "selected_numeric_correlation_heatmap.svg")

**Correlation note:** the highest observed absolute correlations among selected telemetry variables are low in this dataset. Severe multicollinearity is not obvious from the selected correlation matrix, but related variables should still be reviewed during modelling, especially for interpretable linear baselines.

## 11. Component-Specific Analysis

This section summarizes component-specific telemetry patterns without hard-coding conclusions before inspecting the data.

In [ ]:
component_summary = {}
for component, features in {
    "engine": feature_groups["engine"],
    "braking_system": feature_groups["braking_system"],
    "battery_electrical": feature_groups["battery_electrical"],
}.items():
    numeric_features = [f for f in features if f in df.columns and pd.api.types.is_numeric_dtype(df[f])]
    component_summary[component] = df[numeric_features].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
    print(f"\n=== {component.upper()} TELEMETRY SUMMARY ===")
    print(component_summary[component].round(3).to_string())

print("\nComponent-specific EDA interpretation should consider cross-system effects; for example, vehicle operation and environmental fields may still be useful predictors for component-level failures.")

## 12. Data Leakage Assessment

The following fields require special treatment before modelling:

- `failure_type`: likely an outcome label and should not be used as a predictor for imminent-failure targets.
- `failure_date`: likely contains future outcome timing or a no-failure sentinel; it should not be used directly as a predictor for classification.
- Other imminent-failure flags: when predicting one target, the other target-like flags may leak outcome information or encode unsupported label relationships.
- `vehicle_id`: may allow vehicle-specific memorization if random row-level splits are used.
- `timestamp`: can be useful for temporal validation and analysis, but derived time features must be justified by real deployment availability.
- Repeated vehicle observations: require grouped and possibly time-aware validation to avoid overly optimistic estimates.
- `abs_fault_indicator`: may be a legitimate sensor reading, but for brake modelling it should be checked carefully for whether it is already a diagnostic outcome rather than an advance predictor.

In [ ]:
leakage_risk_table = pd.DataFrame([
    {
        "variable": "failure_type",
        "risk": "Directly encodes failure outcome/category.",
        "phase_1_action": "Document as target/outcome candidate, not predictor."
    },
    {
        "variable": "failure_date",
        "risk": "Contains future event timing or no-failure sentinel.",
        "phase_1_action": "Do not use directly as predictor for classification."
    },
    {
        "variable": "engine/brake/battery imminent flags",
        "risk": "Target-like fields can leak labels when predicting another target.",
        "phase_1_action": "Define target formulation before selecting predictors."
    },
    {
        "variable": "vehicle_id",
        "risk": "May allow vehicle-specific memorization under random row split.",
        "phase_1_action": "Consider grouped split by vehicle."
    },
    {
        "variable": "timestamp",
        "risk": "Temporal dependence and possible deployment-time constraints.",
        "phase_1_action": "Consider time-aware validation; justify any derived time fields."
    },
    {
        "variable": "abs_fault_indicator",
        "risk": "May be diagnostic of brake fault rather than advance predictor.",
        "phase_1_action": "Assess real-world timing and availability during feature selection."
    },
])
print(leakage_risk_table.to_string(index=False))

## 13. EDA Conclusions and Modelling Implications

### Major dataset characteristics

- The dataset contains 1,970 records and 41 columns from 50 unique vehicles.
- Each vehicle has repeated telemetry observations, with roughly 35 to 44 observations per vehicle.
- The data contains vehicle metadata, timestamps, engine telemetry, brake telemetry, battery/electrical telemetry, vehicle operation fields, environmental fields, GPS fields, and failure-related labels.

### Data-quality findings

- No missing values were found in the initial inspection.
- No full duplicate rows were found.
- There are duplicate `vehicle_id` + `timestamp` pairs.
- Timestamps are not ordered within vehicles in the file order, so analysis should sort by `vehicle_id` and `timestamp` when temporal sequence matters.
- Some suspicious values require investigation before cleaning decisions: negative wheel speeds, a small number of humidity values above 100, and one slightly negative vibration value.

### Target distributions and class imbalance

- `engine_failure_imminent`: 7 positive records and 1,963 negative records.
- `brake_issue_imminent`: 16 positive records and 1,954 negative records.
- `battery_issue_imminent`: 12 positive records and 1,958 negative records.
- Any imminent issue across the three flags appears in 35 records.
- The imbalance is severe. Accuracy alone will be a weak evaluation metric; precision, recall, and F1-score are essential, and later phases may need additional metrics or imbalance-aware validation.

### Important relationships found

- Engine-positive records show higher engine temperature and vibration and lower oil pressure.
- Brake-positive records show lower brake fluid pressure and higher brake temperature.
- Battery-positive records show lower battery voltage and strongly negative battery current.
- These relationships are descriptive EDA findings, not trained-model conclusions.

### Leakage variables and validation implications

- `failure_type` and `failure_date` should be treated as outcome-related fields, not ordinary predictors for imminent-failure classification.
- If one imminent-failure flag is the target, the other imminent-failure flags should not be automatically used as input features.
- `vehicle_id`, timestamps, and repeated observations create a risk that random row-level splitting will overestimate performance. A grouped split by `vehicle_id` is recommended as the default next-phase validation strategy; time-aware validation should also be considered.

### Correlation and preprocessing implications

- Strong numerical multicollinearity is not obvious from the selected correlation heatmap.
- Categorical fields such as `brand` would require encoding in later phases.
- Numerical fields may require scaling for Logistic Regression.
- Suspicious values should be investigated before any imputation, clipping, or removal decisions.

### Recommendation for modelling formulation

For the next phase, the most defensible initial formulation is to treat failure prediction as a supervised classification problem using carefully selected, leakage-safe telemetry predictors. Because each individual component target has very few positive examples, separate models for `engine_failure_imminent`, `brake_issue_imminent`, and `battery_issue_imminent` may be academically informative but statistically fragile. The next phase should compare the feasibility of:

1. A combined binary target for any imminent critical component issue.
2. Separate component-specific binary targets, reported with clear limitations due to very low positive counts.

The final choice should be made before model training and should be documented with the class balance, validation strategy, and leakage exclusions.